# NB06b — Wilcoxon retraining runs: Yelp (10 paired seeds)

For each of 10 seeds, retrains COA-ANN, Default ANN, COA-CNN+BiLSTM and Default CNN+BiLSTM on Yelp with the same schedule for both arms (ANN ≤50 epochs / patience 5, CNN ≤20 / patience 3; true labels for early stopping) and runs the exact two-sided Wilcoxon signed-rank test. Code identical to the tested Wilcoxon cell of the former NB06, restricted to one dataset so the notebook fits one session.

| | |
|---|---|
| **Required inputs** (Add Input → Your Work → Notebooks; always the newest version) | `01-preprocessing`, `02-coa-ann`, `03-coa-cnn` |
| **Optional inputs** | — |
| **Outputs** | `saved_results/yelp/{ann,cnn}_wilcoxon.json`, `saved_results/yelp/wilcoxon.csv`, `saved_results/units/` |
| **Run time on Kaggle (estimate)** | ≈2 h typical, ≈5 h worst case |
| **Accelerator / Internet** | GPU T4 x2 / off |

NB06a, NB06b and NB06c replace the training part of the former NB06. `06-comparison-new` now only collects their results and makes the tables and figures.

Step-by-step instructions for the whole pipeline: `Code/Newest/RUN_GUIDE.md`. Background: `Decision/012-split-notebooks-for-9h-sessions.md`.

## 0. Configuration

In [ ]:
DATASET = "yelp"   # do not change: one notebook per dataset (06a amazon, 06b yelp, 06c twitter)

In [ ]:
import os, random as _random, warnings, re, string, json, gc, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
SEED       = 42
MAX_VOCAB  = 50000
EMBED_DIM  = 300
DATASETS   = ["amazon", "yelp", "twitter"]

WORKING     = "/kaggle/working"
SHARED_DIR  = f"{WORKING}/shared_data"
RESULTS_DIR = f"{WORKING}/saved_results"
MODELS_DIR  = f"{WORKING}/saved_models"

def ensure_dirs(dataset=None):
    for base in [SHARED_DIR, RESULTS_DIR, MODELS_DIR]:
        os.makedirs(base, exist_ok=True)
        if dataset:
            os.makedirs(f"{base}/{dataset}", exist_ok=True)
        else:
            for ds in DATASETS:
                os.makedirs(f"{base}/{ds}", exist_ok=True)

def set_seeds(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    _random.seed(seed)
    np.random.seed(seed)

set_seeds()
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
    recall_score, matthews_corrcoef, roc_auc_score, average_precision_score,
    classification_report, precision_recall_curve, confusion_matrix)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import wilcoxon

tf.keras.mixed_precision.set_global_policy("mixed_float16")
strategy = tf.distribute.MirroredStrategy()
N_GPUS = strategy.num_replicas_in_sync
print(f"MirroredStrategy: {N_GPUS} GPU(s)")

ANN_BATCH = 1024 * N_GPUS
CNN_BATCH = 128  * N_GPUS
print(f"ANN_BATCH={ANN_BATCH}  CNN_BATCH={CNN_BATCH}")

def set_seeds_tf(seed=SEED):
    set_seeds(seed)
    tf.random.set_seed(seed)

set_seeds_tf()
print(f"TF {tf.__version__}")

ensure_dirs()


In [ ]:
def evaluate(name, y_true, y_pred, y_prob=None):
    auc = round(roc_auc_score(y_true, y_prob), 4) if y_prob is not None else float("nan")
    ap  = round(average_precision_score(y_true, y_prob), 4) if y_prob is not None else float("nan")
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "Model"        : name,
        "Accuracy"     : round(accuracy_score(y_true, y_pred), 4),
        "Precision"    : round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall"       : round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1-score"     : round(f1_score(y_true, y_pred, zero_division=0), 4),
        "Macro-F1"     : round(rep["macro avg"]["f1-score"], 4),
        "Weighted-F1"  : round(rep["weighted avg"]["f1-score"], 4),
        "MCC"          : round(matthews_corrcoef(y_true, y_pred), 4),
        "AUC-ROC"      : auc,
        "Avg-Precision": ap,
    }

def best_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1s = 2*prec*rec/(prec+rec+1e-9)
    idx = np.argmax(f1s[:-1])
    return float(thr[idx])

def make_es(patience=5, verbose=0):
    return EarlyStopping(monitor="val_loss", patience=patience,
                         restore_best_weights=True, verbose=verbose)

def make_tf_ds(X, y, batch, shuffle=True, seed=SEED):
    X = X.astype(np.float32)
    y = y.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle: ds = ds.shuffle(min(len(y), 50000), seed=seed)
    return ds.batch(batch).cache().prefetch(tf.data.AUTOTUNE)

def make_seq_ds(X, y, batch, shuffle=True, seed=SEED):
    X = X.astype(np.int32)
    y = y.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle: ds = ds.shuffle(min(len(y), 50000), seed=seed)
    return ds.batch(batch).cache().prefetch(tf.data.AUTOTUNE)

print("Helpers ready.")


In [ ]:
def build_ann(h1, h2, lr, seed=SEED):
    set_seeds_tf(seed)
    with strategy.scope():
        m = tf.keras.Sequential([
            tf.keras.layers.Dense(h1, activation="relu", input_shape=(EMBED_DIM,)),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(h2, activation="relu"),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32"),
        ])
        m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return m

def build_cnn(filters, lstm_units, lr, vocab_sz, emb_mat, max_len,
               seed=SEED, pretrained=True, use_bilstm=True, use_dropout=True):
    set_seeds_tf(seed)
    with strategy.scope():
        emb = (tf.keras.layers.Embedding(vocab_sz, EMBED_DIM,
                                          weights=[emb_mat], trainable=False)
               if pretrained else
               tf.keras.layers.Embedding(vocab_sz, EMBED_DIM, trainable=True))
        layers = [emb, tf.keras.layers.Conv1D(filters, 3, activation="relu"),
                  tf.keras.layers.MaxPooling1D(2)]
        if use_bilstm:
            layers.append(tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(lstm_units)))
        else:
            layers.append(tf.keras.layers.GlobalMaxPooling1D())
        layers.append(tf.keras.layers.Dense(64, activation="relu"))
        if use_dropout:
            layers.append(tf.keras.layers.Dropout(0.5))
        layers.append(tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32"))
        m = tf.keras.Sequential(layers)
        m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return m

def shallow_ann(h1, lr, seed=SEED):
    set_seeds_tf(seed)
    with strategy.scope():
        m = tf.keras.Sequential([
            tf.keras.layers.Dense(h1, activation="relu", input_shape=(EMBED_DIM,)),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32"),
        ])
        m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return m

print("Model builders ready.")


In [ ]:
def load_dataset(dataset, nb01_shared):
    """Load all arrays for a given dataset."""
    d_dir = f"{nb01_shared}/{dataset}"
    cfg   = json.load(open(f"{d_dir}/config.json"))
    d = {
        "name"         : dataset,
        "cfg"          : cfg,
        "max_len"      : cfg["max_len"],
        "vocab_size"   : cfg["vocab_size"],
        "X_emb_train"  : np.load(f"{d_dir}/X_emb_train.npy"),
        "X_emb_val"    : np.load(f"{d_dir}/X_emb_val.npy"),
        "X_emb_test"   : np.load(f"{d_dir}/X_emb_test.npy"),
        "X_seq_train"  : np.load(f"{d_dir}/X_seq_train.npy"),
        "X_seq_val"    : np.load(f"{d_dir}/X_seq_val.npy"),
        "X_seq_test"   : np.load(f"{d_dir}/X_seq_test.npy"),
        "y_train"      : np.load(f"{d_dir}/y_train.npy"),
        "y_val"        : np.load(f"{d_dir}/y_val.npy"),
        "y_test"       : np.load(f"{d_dir}/y_test.npy"),
        "emb_matrix"   : np.load(f"{d_dir}/emb_matrix.npy"),
        "class_weights": {int(k):float(v) for k,v in
                          json.load(open(f"{d_dir}/class_weights.json")).items()},
    }
    with open(f"{d_dir}/test_texts_raw.txt", encoding="utf-8") as f:
        d["test_texts_raw"] = f.read().splitlines()
    with open(f"{d_dir}/X_txt_train.txt", encoding="utf-8") as f:
        d["X_txt_train"] = f.read().splitlines()
    with open(f"{d_dir}/X_txt_val.txt", encoding="utf-8") as f:
        d["X_txt_val"] = f.read().splitlines()
    with open(f"{d_dir}/X_txt_test.txt", encoding="utf-8") as f:
        d["X_txt_test"] = f.read().splitlines()
    print(f"  {dataset}: train={len(d['y_train']):,} val={len(d['y_val']):,} "
          f"test={len(d['y_test']):,} max_len={d['max_len']}")
    return d

def tv_combined(d):
    """Train+val combined for final training."""
    return {
        "X_emb": np.vstack([d["X_emb_train"], d["X_emb_val"]]),
        "X_seq": np.vstack([d["X_seq_train"], d["X_seq_val"]]),
        "y"    : np.concatenate([d["y_train"], d["y_val"]]),
        "X_txt": d["X_txt_train"] + d["X_txt_val"],
    }

print("Loader ready.")


## 1. Inputs

In [ ]:
# ── Inputs ────────────────────────────────────────────────────────────────────
NB01_DIR    = "/kaggle/input/notebooks/abhaysharma01702/01-preprocessing"
NB01_SHARED = f"{NB01_DIR}/shared_data"
NB02_RESULTS = "/kaggle/input/notebooks/abhaysharma01702/02-coa-ann/saved_results"
NB03_RESULTS = "/kaggle/input/notebooks/abhaysharma01702/03-coa-cnn/saved_results"
for _name, _p in [("01-preprocessing", f"{NB01_SHARED}/{DATASET}/config.json"),
                  ("02-coa-ann", f"{NB02_RESULTS}/{DATASET}/coa_ann.json"),
                  ("03-coa-cnn", f"{NB03_RESULTS}/{DATASET}/coa_cnn.json")]:
    print(f"  {_name:18s} {'✓' if os.path.exists(_p) else '✗ MISSING — add this notebook as an input'}")


In [ ]:
# ── Run control ───────────────────────────────────────────────────────────────
# This notebook is sized to finish well inside one 9-hour Kaggle session (see the header). As a safety net it saves
# every finished unit (one model/fold/seed) immediately and stops cleanly after TIME_LIMIT_HOURS. If that ever
# happens: save the version, add it as an input, list its slug in RESUME_SLUGS and run again — finished units are
# copied over and skipped.
import time, glob, shutil
RESUME_SLUGS     = []          # e.g. ["06b-wilcoxon-yelp"]
TIME_LIMIT_HOURS = 8.0         # Kaggle sessions end at 9 h; the last hour is margin for loading and final cells
T_START = time.time()
def hours_used(): return (time.time() - T_START) / 3600

UNIT_DIR = f"{RESULTS_DIR}/units"
os.makedirs(UNIT_DIR, exist_ok=True)
for _slug in RESUME_SLUGS:
    _n = 0
    for _f in glob.glob(f"/kaggle/input/notebooks/abhaysharma01702/{_slug}/saved_results/units/*.json"):
        _dst = f"{UNIT_DIR}/{os.path.basename(_f)}"
        if not os.path.exists(_dst):
            shutil.copy(_f, _dst); _n += 1
    print(f"Resumed {_n} finished units from {_slug}")

def unit_done(key): return os.path.exists(f"{UNIT_DIR}/{key}.json")
def unit_load(key):
    with open(f"{UNIT_DIR}/{key}.json") as fh:
        return json.load(fh)
def unit_save(key, obj):
    """Atomic write: a session killed mid-write cannot leave a truncated unit file behind."""
    tmp = f"{UNIT_DIR}/{key}.json.tmp"
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=2)
    os.replace(tmp, f"{UNIT_DIR}/{key}.json")

class TimeLimit(Exception):
    pass
def check_time():
    if hours_used() > TIME_LIMIT_HOURS:
        raise TimeLimit()

STAT_SEEDS = [42, 7, 13, 99, 2024, 11, 23, 57, 101, 314]   # 10 paired seeds
DATASETS = [DATASET]                                         # this notebook runs one dataset


## 2. Wilcoxon retraining runs and test

In [ ]:
# ── Wilcoxon signed-rank test over paired seeds ───────────────────────────────
# 2026-10-12: early stopping now uses the true validation labels (was np.zeros); 10 seeds instead of 5.
# Both arms use the same schedule (ANN ≤50 epochs / patience 5, CNN ≤20 / patience 3) and the same seed offsets as before.
def run_model(m, X_tr, y_tr, X_vl, y_vl, X_te, y_te, cw, is_cnn=False):
    if is_cnn:
        m.fit(make_seq_ds(X_tr, y_tr, CNN_BATCH),
              validation_data=make_seq_ds(X_vl, y_vl, CNN_BATCH, shuffle=False),
              epochs=20, callbacks=[make_es(3)], class_weight=cw, verbose=0)
        p = m.predict(make_seq_ds(X_te, np.zeros(len(y_te)), CNN_BATCH, shuffle=False), verbose=0).flatten()
    else:
        m.fit(make_tf_ds(X_tr, y_tr, ANN_BATCH),
              validation_data=make_tf_ds(X_vl, y_vl, ANN_BATCH, shuffle=False),
              epochs=50, callbacks=[make_es(5)], class_weight=cw, verbose=0)
        p = m.predict(make_tf_ds(X_te, np.zeros(len(y_te)), ANN_BATCH, shuffle=False), verbose=0).flatten()
    return float(f1_score(y_te, (p > 0.5).astype(int)))

try:
    for ds in DATASETS:
        todo = [s for s in STAT_SEEDS if not unit_done(f"wil_{ds}_s{s}")]
        if not todo:
            continue
        print(f"\n{'='*60}\nWILCOXON | {ds.upper()}  ({len(todo)} seeds to run)\n{'='*60}")
        d = load_dataset(ds, NB01_SHARED)
        ml=d["max_len"]; vs=d["vocab_size"]; emb=d["emb_matrix"]
        ann_cfg=json.load(open(f"{NB02_RESULTS}/{ds}/coa_ann.json"))
        cnn_cfg=json.load(open(f"{NB03_RESULTS}/{ds}/coa_cnn.json"))
        h1=ann_cfg["best_h1"]; h2=ann_cfg["best_h2"]; ann_lr=ann_cfg["best_lr"]
        f_=cnn_cfg["best_filters"]; lstm_u=cnn_cfg["best_lstm_units"]; cnn_lr=cnn_cfg["best_lr"]
        tv = tv_combined(d)
        cw_tv=dict(zip([int(k) for k in np.unique(tv["y"])],
                       [float(v) for v in compute_class_weight("balanced", classes=np.unique(tv["y"]), y=tv["y"])]))
        n_es=int(len(tv["y"])*0.1)
        Xe, Xs, y = tv["X_emb"], tv["X_seq"], tv["y"]
        for seed in todo:
            check_time()
            r = {"seed": seed}
            set_seeds_tf(seed)
            r["coa_ann"] = run_model(build_ann(h1, h2, ann_lr, seed), Xe[:-n_es], y[:-n_es], Xe[-n_es:], y[-n_es:],
                                     d["X_emb_test"], d["y_test"], cw_tv)
            set_seeds_tf(seed + 500)
            r["def_ann"] = run_model(build_ann(128, 64, 1e-3, seed + 500), Xe[:-n_es], y[:-n_es], Xe[-n_es:], y[-n_es:],
                                     d["X_emb_test"], d["y_test"], cw_tv)
            set_seeds_tf(seed + 1000)
            r["coa_cnn"] = run_model(build_cnn(f_, lstm_u, cnn_lr, vs, emb, ml, seed + 1000), Xs[:-n_es], y[:-n_es],
                                     Xs[-n_es:], y[-n_es:], d["X_seq_test"], d["y_test"], cw_tv, is_cnn=True)
            set_seeds_tf(seed + 1500)
            r["def_cnn"] = run_model(build_cnn(128, 64, 1e-3, vs, emb, ml, seed + 1500), Xs[:-n_es], y[:-n_es],
                                     Xs[-n_es:], y[-n_es:], d["X_seq_test"], d["y_test"], cw_tv, is_cnn=True)
            unit_save(f"wil_{ds}_s{seed}", r)
            print(f"  Seed {seed}: COA-ANN={r['coa_ann']:.4f} Def-ANN={r['def_ann']:.4f}  "
                  f"COA-CNN={r['coa_cnn']:.4f} Def-CNN={r['def_cnn']:.4f}   [{hours_used():.2f} h]")
            tf.keras.backend.clear_session(); gc.collect()
        del d, tv, emb, Xe, Xs, y; gc.collect()
except TimeLimit:
    print(f"\nTime limit ({TIME_LIMIT_HOURS} h) reached. Finished seeds are saved; "
          "link this version, set RESUME_SLUGS and run again.")

# Statistics on whatever seeds are finished (n_seeds is recorded with every result)
wilcoxon_all = {}
for ds in DATASETS:
    runs = [unit_load(f"wil_{ds}_s{s}") for s in STAT_SEEDS if unit_done(f"wil_{ds}_s{s}")]
    if len(runs) < 2:
        print(f"{ds}: not enough finished seeds yet"); continue
    if len(runs) < len(STAT_SEEDS):
        print(f"WARNING {ds}: only {len(runs)}/{len(STAT_SEEDS)} seeds finished — do not report these p-values")
    r = {}
    for mt in ["ann", "cnn"]:
        coa = [x[f"coa_{mt}"] for x in runs]; dfl = [x[f"def_{mt}"] for x in runs]
        diff = np.array(coa) - np.array(dfl)
        p = float(wilcoxon(coa, dfl).pvalue) if np.any(diff != 0) else 1.0
        r[mt] = {"seeds": [x["seed"] for x in runs], "n_seeds": len(runs), "coa_f1": coa, "def_f1": dfl,
                 "coa_mean": round(float(np.mean(coa)), 4), "def_mean": round(float(np.mean(dfl)), 4),
                 "coa_std": round(float(np.std(coa)), 4), "def_std": round(float(np.std(dfl)), 4),
                 "mean_diff": round(float(diff.mean()), 4), "coa_wins": int((diff > 0).sum()),
                 "p_value": round(p, 6), "significant": bool(p < 0.05)}
        print(f"  {ds} {mt.upper()}: COA={r[mt]['coa_mean']}±{r[mt]['coa_std']}  "
              f"Def={r[mt]['def_mean']}±{r[mt]['def_std']}  wins={r[mt]['coa_wins']}/{len(runs)}  p={p:.4f}")
    wilcoxon_all[ds] = r
    r_dir = f"{RESULTS_DIR}/{ds}"; os.makedirs(r_dir, exist_ok=True)
    json.dump(r["ann"], open(f"{r_dir}/ann_wilcoxon.json", "w"), indent=2)
    json.dump(r["cnn"], open(f"{r_dir}/cnn_wilcoxon.json", "w"), indent=2)
    pd.DataFrame({"Seed": [str(x["seed"]) for x in runs],
                  "COA-ANN": [round(x["coa_ann"], 4) for x in runs], "Def-ANN": [round(x["def_ann"], 4) for x in runs],
                  "COA-CNN": [round(x["coa_cnn"], 4) for x in runs], "Def-CNN": [round(x["def_cnn"], 4) for x in runs],
                 }).to_csv(f"{r_dir}/wilcoxon.csv", index=False)

print("\nWilcoxon cell finished.")
